# Glossary Formatter

This notebook parses `data/glossary.md` into a consolidated glossary table and computes in-notebook summaries/visualizations.

Exports:
- `outputs/glossary/glossary.tsv` (with `source_domain` and `tier` columns)
- `outputs/glossary/glossary_tier_summary.txt`

---

## New Implementation — Parsing `glossary.md`

Parses `data/glossary.md` and writes a single consolidated TSV:

- `outputs/glossary/glossary.tsv`

Summary statistics and the example-share chart are computed below without writing additional files.


In [ ]:
from __future__ import annotations

import re
from pathlib import Path
from urllib.parse import unquote

import matplotlib.pyplot as plt
import pandas as pd

WORKDIR = Path.cwd()
if not (WORKDIR / 'data').exists():
    WORKDIR = WORKDIR.parent
GLOSSARY_MD_PATH = WORKDIR / 'data' / 'glossary.md'
OUTPUT_DIR = WORKDIR / 'outputs' / 'glossary'
OUTPUT_TSV = OUTPUT_DIR / 'glossary.tsv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GLOSSARY_MD_PATH


In [ ]:
def _extract_link_url(text: str) -> str:
    m = re.search(r'\]\((https?://[^)]+)\)', text)
    return unquote(m.group(1).strip()) if m else ''


def _parse_label(block: str, label: str) -> str:
    m = re.search(rf'_{re.escape(label)}_:\s*(.*?)(?=\s*_[^_]+_:|$)', block, re.S)
    if not m:
        return ''
    return re.sub(r'\s+', ' ', m.group(1)).strip()


def parse_glossary_md(path: Path) -> pd.DataFrame:
    raw = path.read_text(encoding='utf-8', errors='ignore')
    # Normalize backslash line-continuations (soft-wrap artifacts from HTML source)
    raw = re.sub(r'\\\n\s*', '\n', raw)

    # Split into per-term blocks at each **term** heading
    entries = re.split(r'\n(?=\*\*)', raw)

    rows: list[dict] = []
    for entry in entries:
        term_m = re.match(r'\*\*([^*]+)\*\*', entry.strip())
        if not term_m:
            continue
        term = term_m.group(1).strip()

        sf_m = re.search(r'_Surface forms_:\s*([^\n]+)', entry)
        surface_forms = sf_m.group(1).strip() if sf_m else ''

        persona_block_m = re.search(
            r'_Persona/In-Group_:.*?(?=\nDescription|\nExample|\Z)', entry, re.S
        )
        persona_block = persona_block_m.group(0) if persona_block_m else ''
        persona = _parse_label(persona_block, 'Persona/In-Group')
        covert_meaning = _parse_label(persona_block, 'Covert (in-group) meaning')
        dogwhistle_type = _parse_label(persona_block, 'Type')
        register = _parse_label(persona_block, 'Register')

        desc_header_m = re.search(r'Description \(from [^\n]+', entry)
        description_source = _extract_link_url(desc_header_m.group(0)) if desc_header_m else ''
        desc_body_m = re.search(
            r'Description \(from [^\n]+\n(.*?)(?=\nExample context|\Z)', entry, re.S
        )
        description = (
            re.sub(r'\s+', ' ', desc_body_m.group(1)).strip() if desc_body_m else ''
        )

        examples, ex_sources, ex_speakers, ex_dates = [], [], [], []
        for ex_m in re.finditer(
            r'Example context \(in ([^\n]+)\n(.*?)(?=\nExample context|\Z)', entry, re.S
        ):
            source_url = _extract_link_url(ex_m.group(1))
            body = ex_m.group(2)
            speaker = _parse_label(body, 'Speaker')
            date = _parse_label(body, 'Date')
            ex_text = re.sub(r'_Speaker_:.*', '', body, flags=re.S).strip()
            ex_text = re.sub(r'\s+', ' ', ex_text).strip()
            if ex_text:
                examples.append(ex_text)
                ex_sources.append(source_url)
                ex_speakers.append(speaker)
                ex_dates.append(date)

        rows.append(
            {
                'term': term,
                'surface_forms': surface_forms,
                'persona_in_group': persona,
                'covert_meaning': covert_meaning,
                'type': dogwhistle_type,
                'register': register,
                'description': description,
                'description_source': description_source,
                'example_count': len(examples),
                'examples': ' || '.join(examples),
                'example_sources': ' ; '.join(s for s in ex_sources if s),
                'example_speakers': ' ; '.join(s for s in ex_speakers if s),
                'example_dates': ' ; '.join(s for s in ex_dates if s),
            }
        )

    return pd.DataFrame(rows)


glossary_df = parse_glossary_md(GLOSSARY_MD_PATH)
glossary_df.to_csv(OUTPUT_TSV, sep='\t', index=False)

print(f'Wrote {len(glossary_df):,} term rows  →  {OUTPUT_TSV}')
glossary_df.head(5)


---

## Provenance Tier Classification

Classifies each entry by the credibility tier of its description source domain:

| Tier | Domain types |
|------|-------------|
| **1** | Peer-reviewed journals, major news outlets (NYT/WaPo/Guardian/Vox/FiveThirtyEight/Slate), civil-rights watchdogs (ADL/AJC/NCCM), institutional/academic sources |
| **2** | Advocacy orgs, progressive media, subject-matter blogs — credible but below Tier 1 standards |
| **3** | Crowd-edited wikis (RationalWiki, Wikipedia) — high coverage, lower sourcing rigour |
| **unknown** | Domains not in any tier set; printed as warnings for manual review |

Adds `source_domain` and `tier` to the in-memory `glossary_df` and overwrites `glossary.tsv`.

In [ ]:
from urllib.parse import urlparse

# Tier 1: peer-reviewed journals, major mainstream outlets, civil-rights orgs,
# and institutional/academic sources with editorial oversight.
TIER_1_DOMAINS = {
    "link.springer.com", "www.tandfonline.com", "www.taylorfrancis.com", "brill.com",
    "www.nytimes.com", "www.washingtonpost.com", "www.theguardian.com", "www.vox.com",
    "fivethirtyeight.com", "slate.com", "prospect.org", "foreignpolicy.com",
    "www.adl.org", "www.ajc.org", "www.nccm.ca", "antisemitism.org.uk",
    "ianhaneylopez.com", "contemporaryrhetoric.com", "today.tamu.edu",
    "www.law.cuny.edu", "s-usih.org"
}

# Tier 2: advocacy orgs, progressive media, subject-matter blogs — credible
# but without Tier 1 editorial standards or peer review.
TIER_2_DOMAINS = {
    "colorofchange.org", "queervegan.com", "everydayfeminism.com",
    "thedemlabs.org", "blmgrassroots.org", "medium.com", "helenldecruz.medium.com",
    "www.theroot.com", "theconversation.com", "politicalresearch.org",
    "forward.com", "www.dailykos.com", "talkingpointsmemo.com",
    "www.salon.com", "nymag.com", "www.sapiens.org", "www.patheos.com",
    "www.usnews.com", "www.newsweek.com", "www.thedailybeast.com",
    "www.csmonitor.com", "www.latimes.com", "www.haaretz.com",
    "www.jta.org", "storyful.com", "hyperallergic.com", "money.cnn.com",
    "www.ourspectrum.com", "www.cpreview.org", "electionsos.com",
    "jacksonfreepress.com", "papers.ssrn.com"
}

# Tier 3: crowd-edited wikis — broad coverage but lower sourcing rigour.
TIER_3_DOMAINS = {"rationalwiki.org", "en.wikipedia.org"}

# These entries link to google.com/books, verified to resolve to
# López (2014) "Dog Whistle Politics" (OUP) — treated as Tier 1.
TIER_1_OVERRIDES = {
    "affirmative action", "gangbanger", "freedom of association", "food stamp president",
}


def _extract_domain(url: str) -> str:
    if not url:
        return ""
    try:
        return urlparse(url.strip()).netloc.lower()
    except Exception:
        return ""


def _classify_tier(term: str, domain: str) -> str:
    if not domain:
        return "unknown"
    if domain in TIER_1_DOMAINS:
        return "1"
    if domain in TIER_2_DOMAINS:
        return "2"
    if domain in TIER_3_DOMAINS:
        return "3"
    if domain == "www.google.com" and term in TIER_1_OVERRIDES:
        return "1"
    return "unknown"

In [ ]:
OUTPUT_SUMMARY = OUTPUT_DIR / 'glossary_tier_summary.txt'

glossary_df['source_domain'] = glossary_df['description_source'].map(_extract_domain)
glossary_df['tier'] = glossary_df.apply(
    lambda row: _classify_tier(row['term'], row['source_domain']), axis=1
)

# Warn on genuinely unknown domains (non-empty URL that matched no tier)
unknown_mask = (glossary_df['tier'] == 'unknown') & (glossary_df['source_domain'] != '')
for dom in sorted(glossary_df.loc[unknown_mask, 'source_domain'].unique()):
    terms = glossary_df.loc[glossary_df['source_domain'] == dom, 'term'].tolist()
    print(f"WARNING: unknown domain '{dom}' — {len(terms)} term(s): {', '.join(terms)}")

glossary_df.to_csv(OUTPUT_TSV, sep='\t', index=False)
print(f'Updated {len(glossary_df):,} rows → {OUTPUT_TSV}')

# --- Summary report ---
lines = ['=' * 60, 'GLOSSARY PROVENANCE TIER SUMMARY', '=' * 60, '']

tier_counts = glossary_df['tier'].value_counts().reindex(['1', '2', '3', 'unknown'], fill_value=0)
lines += ['Total entries per tier', '-' * 30]
for tier, count in tier_counts.items():
    lines.append(f'  Tier {tier}: {count:>4d} entries')
lines += [f'  TOTAL  : {len(glossary_df):>4d} entries', '']

tier3_df = glossary_df[glossary_df['tier'] == '3']
lines += ['Tier 3 — domain breakdown', '-' * 30]
for dom, cnt in tier3_df['source_domain'].value_counts().items():
    lines.append(f'  {dom}: {cnt}')
lines.append('')

unknown_df = glossary_df[glossary_df['tier'] == 'unknown']
lines += ['Unknown — domain breakdown', '-' * 30]
for dom, cnt in unknown_df['source_domain'].value_counts().items():
    lines.append(f'  {dom if dom else "(no URL)"}: {cnt}')
lines.append('')

lines += ['Per persona/in-group × tier entry counts', '-' * 50]
exploded = glossary_df.copy()
exploded['persona_split'] = exploded['persona_in_group'].str.split(' / ')
exploded = exploded.explode('persona_split')
exploded['persona_split'] = exploded['persona_split'].str.strip().replace('', 'unknown')
pivot = (
    exploded.groupby(['persona_split', 'tier']).size()
    .unstack(fill_value=0)
    .reindex(columns=['1', '2', '3', 'unknown'], fill_value=0)
)
pivot['total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('total', ascending=False)

col_header = f"  {'Persona/In-Group':<35s}  {'T1':>4}  {'T2':>4}  {'T3':>4}  {'?':>4}  {'tot':>4}"
lines += [col_header, '  ' + '-' * (len(col_header) - 2)]
for persona, row in pivot.iterrows():
    lines.append(
        f"  {str(persona):<35s}  "
        f"{row.get('1', 0):>4d}  {row.get('2', 0):>4d}  "
        f"{row.get('3', 0):>4d}  {row.get('unknown', 0):>4d}  {row['total']:>4d}"
    )

report = '\n'.join(lines)
OUTPUT_SUMMARY.write_text(report, encoding='utf-8')
print(f'Wrote summary → {OUTPUT_SUMMARY}')
print()
print(report)

In [ ]:
# --- Group metrics summary ---
group_cols = ['persona_in_group', 'type', 'register']
group_metrics = (
    glossary_df.groupby(group_cols, dropna=False, as_index=False)
    .agg(
        term_count=('term', 'count'),
        unique_term_count=('term', 'nunique'),
        total_examples=('example_count', 'sum'),
        avg_examples_per_term=('example_count', 'mean'),
    )
    .sort_values(['total_examples', 'term_count'], ascending=False)
)
group_metrics['avg_examples_per_term'] = group_metrics['avg_examples_per_term'].round(3)

print(f'{len(group_metrics):,} persona/type/register groups')
group_metrics.head(15)


In [ ]:
# --- Example-share pie chart by persona/in-group ---
examples_by_group = (
    glossary_df.groupby('persona_in_group', dropna=False)['example_count']
    .sum()
    .sort_values(ascending=False)
)

examples_by_group.index = [
    idx if isinstance(idx, str) and idx.strip() else 'Unknown'
    for idx in examples_by_group.index
]

plt.figure(figsize=(9, 9))
plt.pie(
    examples_by_group.values,
    labels=examples_by_group.index,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.8,
)
plt.title('Share of Example Contexts by Persona/In-Group')
plt.tight_layout()
plt.show()

examples_by_group.to_frame('example_count')
